In [2]:
import pandas as pd

df = pd.read_csv("final set for prediction.csv")

columns_to_keep = [
    "winner",
    "cumulative_runs",
    "cumulative_wickets",
    "current_run_rate",
    "required_run_rate",
    "inning",
    "id",
    "batting_team",
    "bowling_team",
    "venue",
]

try:
    df_filtered = df[columns_to_keep]
    print("Unwanted columns removed successfully.")
    print(df_filtered.head(10))

except KeyError as e:
    print(f"Error removing columns: {e}")

csv_file_name = "filtered_cricket_data.csv"
df_filtered.to_csv(csv_file_name, index=False)
print(f"Filtered data saved to {csv_file_name}")

<ipython-input-2-e757881b13df>:4: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("final set for prediction.csv")


Unwanted columns removed successfully.
                  winner  cumulative_runs  cumulative_wickets  \
0  Kolkata Knight Riders                1                   0   
1  Kolkata Knight Riders                1                   0   
2  Kolkata Knight Riders                2                   0   
3  Kolkata Knight Riders                2                   0   
4  Kolkata Knight Riders                2                   0   
5  Kolkata Knight Riders                2                   0   
6  Kolkata Knight Riders                3                   0   
7  Kolkata Knight Riders                3                   0   
8  Kolkata Knight Riders                7                   0   
9  Kolkata Knight Riders               11                   0   

   current_run_rate required_run_rate  inning      id           batting_team  \
0              0.00               NaN       1  335982  Kolkata Knight Riders   
1              6.00               NaN       1  335982  Kolkata Knight Riders   
2    

In [3]:
import pandas as pd

df = pd.read_csv("filtered_cricket_data.csv")

first_innings_targets = (
    df[df["inning"] == 1]
    .groupby("id")["cumulative_runs"]
    .max()
    .rename("target")
)

df = df.merge(first_innings_targets, on="id", how="left")

df["target"] = df["target"].where(df["inning"] == 2, 0)

df["win"] = (df["batting_team"] == df["winner"]).astype(int)
output_file = "filtered_cricket_data_with_target_win.csv"
df.to_csv(output_file, index=False)

print(f"Updated dataset saved as '{output_file}'")

<ipython-input-3-953b56dd8bb1>:4: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("filtered_cricket_data.csv")


Updated dataset saved as 'filtered_cricket_data_with_target_win.csv'


In [4]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("filtered_cricket_data_with_target_win.csv")

categorical_columns = ["winner", "batting_team", "bowling_team", "venue"]

label_encoders = {col: LabelEncoder() for col in categorical_columns}
for col in categorical_columns:
    df[col] = label_encoders[col].fit_transform(df[col])

df["required_run_rate"] = pd.to_numeric(df["required_run_rate"], errors="coerce").fillna(0)

output_file = "encoded_cricket_data.csv"
df.to_csv(output_file, index=False)

print(df.head(10))
print(f"Encoded dataset saved to: {output_file}")

<ipython-input-4-a9289908d5d6>:5: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("filtered_cricket_data_with_target_win.csv")


   winner  cumulative_runs  cumulative_wickets  current_run_rate  \
0       8                1                   0              0.00   
1       8                1                   0              6.00   
2       8                2                   0              6.00   
3       8                2                   0              4.00   
4       8                2                   0              3.00   
5       8                2                   0              2.40   
6       8                3                   0              3.00   
7       8                3                   0              3.00   
8       8                7                   0              6.00   
9       8               11                   0              8.25   

   required_run_rate  inning      id  batting_team  bowling_team  venue  \
0                0.0       1  335982             8            16     17   
1                0.0       1  335982             8            16     17   
2                0.0      

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("encoded_cricket_data.csv")

features = [
    "batting_team", "bowling_team", "inning",
    "cumulative_wickets", "current_run_rate",
    "required_run_rate", "target"
]
target = "win"

X_train, X_test, y_train, y_test = train_test_split(
    df[features], df[target], test_size=0.3, random_state=42
)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (182644, 7)
Testing data shape: (78276, 7)


In [7]:
# Save train and test datasets
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Train and test datasets saved successfully!")

Train and test datasets saved successfully!
